In [48]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import string
import pandas as pd

In [ ]:
URL = 'https://books.toscrape.com/catalogue/'
response = requests.get(URL)

In [3]:
soup = BeautifulSoup(response.text, 'html.parser')

In [4]:
print(soup.prettify())

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   All products | Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:29" name="created"/>
  <meta content="" name="description"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="NOARCHIVE,NOCACHE" name="robots"/>
  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
  <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
  <link href="static/oscar/favicon.ico" rel="shortcut icon"/>
  <link href="static/oscar/css/styles.css" rel="stylesheet" type="tex

In [8]:
raw_books = soup.findAll('article', class_ = 'product_pod')

C:\Users\mmoed\AppData\Local\Temp\ipykernel_17668\2504786083.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  raw_books = soup.findAll('article', class_ = 'product_pod')


In [ ]:
print(len(raw_books))

20


In [37]:
"Mohammed: Talal".translate(str.maketrans('','',string.punctuation))

'Mohammed Talal'

In [61]:
Books_list = []
page_num = 1
url = URL
while url and page_num <= 5:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    raw_books = soup.findAll('article', class_ = 'product_pod')
    for item in raw_books:
        name = item.find('h3').find('a')['title']
        rating = item.find('p', class_= 'star-rating')['class'][1]
        price = item.find('p', class_= 'price_color').text
        image_src = item.find('img')['src']
        cleaned_name = name.translate(str.maketrans('','',string.punctuation))
        # Downlaod images
        image_url = urljoin(URL,image_src)
        img_content = requests.get(image_url).content
        with open('../images/' + cleaned_name.replace(" ", "_")[:30] + '.png', "wb") as file:
            file.write(img_content)
        Books_list.append({
            "name": name,
            "rating": rating,
            "price": price,
            "img_url": image_url,
        })
    nxt_btn = soup.find('li', class_ = 'next')
    page_num += 1
    url =f'https://books.toscrape.com/catalogue/page-{page_num}.html' if nxt_btn else None
    


C:\Users\mmoed\AppData\Local\Temp\ipykernel_17668\434006643.py:7: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  raw_books = soup.findAll('article', class_ = 'product_pod')


KeyboardInterrupt: 

In [59]:
df = pd.DataFrame(Books_list)

In [54]:
df.to_csv('../data/raw_data.csv')

In [63]:
df = pd.read_csv('../data/raw_data.csv')
df.sample(10)

,Unnamed: 0,name,rating,price,img_url
21,21,How Music Works,Two,Â£37.32,https://books.toscrape.com/media/cache/5c/c8/5...
70,70,The Art Forger,Three,Â£40.76,https://books.toscrape.com/media/cache/70/fa/7...
76,76,"Saga, Volume 6 (Saga (Collected Editions) #6)",Three,Â£25.02,https://books.toscrape.com/media/cache/43/ae/4...
47,47,Untitled Collection: Sabbath Poems 2014,Four,Â£14.27,https://books.toscrape.com/media/cache/f9/3b/f...
19,19,It's Only the Himalayas,Two,Â£45.17,https://books.toscrape.com/media/cache/27/a5/2...
37,37,"In a Dark, Dark Wood",One,Â£19.63,https://books.toscrape.com/media/cache/23/85/2...
25,25,Birdsong: A Story in Pictures,Three,Â£54.64,https://books.toscrape.com/media/cache/af/6e/a...
16,16,Olio,One,Â£23.88,https://books.toscrape.com/media/cache/55/33/5...
84,84,Patience,Three,Â£10.16,https://books.toscrape.com/media/cache/01/72/0...
44,44,Without Borders (Wanderlove #1),Two,Â£45.07,https://books.toscrape.com/media/cache/24/e2/2...


In [69]:
df['price'] = df['price'].str.replace('Â£', '')
df['price'] = df['price'].astype(float)

In [75]:
df['rating'] = df['rating'].map({'Three': 3,
                   'One': 1,
                    'Four' : 4,
                    'Five': 5,
                    'Two': 2
                    })

In [76]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  100 non-null    int64  
 1   name        100 non-null    str    
 2   rating      100 non-null    int64  
 3   price       100 non-null    float64
 4   img_url     100 non-null    str    
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 KB


In [ ]:
df.sort_values(by=['rating'], inplace=)

,Unnamed: 0,name,rating,price,img_url
1,1,Tipping the Velvet,1,53.74,https://books.toscrape.com/media/cache/26/0c/2...
2,2,Soumission,1,50.10,https://books.toscrape.com/media/cache/3e/ef/3...
5,5,The Requiem Red,1,22.65,https://books.toscrape.com/media/cache/68/33/6...
9,9,The Black Maria,1,52.15,https://books.toscrape.com/media/cache/58/46/5...
17,17,Mesaerion: The Best Science Fiction Stories 18...,1,37.59,https://books.toscrape.com/media/cache/09/a3/0...
...,...,...,...,...,...
72,72,The Activist's Tao Te Ching: Ancient Advice fo...,5,32.24,https://books.toscrape.com/media/cache/96/db/9...
80,80,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",5,13.61,https://books.toscrape.com/media/cache/78/0b/7...
81,81,Princess Between Worlds (Wide-Awake Princess #5),5,13.34,https://books.toscrape.com/media/cache/b7/e8/b...
65,65,The Inefficiency Assassin: Time Management Tac...,5,20.59,https://books.toscrape.com/media/cache/75/dc/7...


In [78]:
df[df.rating == 1]

,Unnamed: 0,name,rating,price,img_url
1,1,Tipping the Velvet,1,53.74,https://books.toscrape.com/media/cache/26/0c/2...
2,2,Soumission,1,50.10,https://books.toscrape.com/media/cache/3e/ef/3...
5,5,The Requiem Red,1,22.65,https://books.toscrape.com/media/cache/68/33/6...
9,9,The Black Maria,1,52.15,https://books.toscrape.com/media/cache/58/46/5...
16,16,Olio,1,23.88,https://books.toscrape.com/media/cache/55/33/5...
17,17,Mesaerion: The Best Science Fiction Stories 18...,1,37.59,https://books.toscrape.com/media/cache/09/a3/0...
20,20,In Her Wake,1,12.84,https://books.toscrape.com/media/cache/5d/72/5...
33,33,The Bear and the Piano,1,36.89,https://books.toscrape.com/media/cache/cf/bb/c...
37,37,"In a Dark, Dark Wood",1,19.63,https://books.toscrape.com/media/cache/23/85/2...
45,45,When We Collided,1,31.77,https://books.toscrape.com/media/cache/08/04/0...
